In [8]:
import ffmpeg
import subprocess
import json
from tqdm import tqdm
import time
import math 
import cv2
import numpy as np
import holoviews as hv
import panel as pn
from holoviews import streams

In [11]:
# Specify the video file path
video_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20.mp4"
output_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20_rotated_angletest_padded.mp4"
#input angle to rotate video 
angleToRotate = 30

In [12]:
# Use ffmpeg to retrieve the metadata
try:
    # Run ffmpeg to get the file's metadata
    probe = ffmpeg.probe(video_file)
    
    # Extract the codec type for the video stream
    video_stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'video'), None)
    
    if video_stream:
        codec_name = video_stream.get('codec_name', 'Unknown')
        print(f"Video codec: {codec_name}")
    else:
        print("No video stream found.")
        
except ffmpeg.Error as e:
    print("An error occurred while reading the video file:", e)

input_file = video_file

# Get the video dimensions and duration
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])
duration = float(video_info['duration'])

# Calculate new dimensions to fit the rotated video
angle_rad = math.radians(angleToRotate)
new_width = int(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
new_height = int(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))

# Define the ffmpeg command to add padding, rotate, and output with JPEG compression
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('pad', new_width, new_height, (new_width - width) // 2, (new_height - height) // 2, color='0xFFFFFF')
    .filter('rotate', str(angle_rad))
    .output(output_file, vcodec='mjpeg', vsync='vfr')
    .global_args('-progress', 'pipe:1', '-nostats')  # Enable progress output
)

# Run ffmpeg as a subprocess and track progress
process = ffmpeg_command.run_async(pipe_stdout=True, pipe_stderr=True, overwrite_output=True)
pbar = tqdm(total=duration, desc="Processing Video", unit="s", dynamic_ncols=True)

# Track ffmpeg progress
for line in process.stderr:
    line = line.decode('utf-8').strip()
    if "out_time_ms" in line:
        out_time_ms = int(line.split('=')[1].strip())
        current_time = out_time_ms / 1_000_000  # Convert to seconds
        pbar.update(current_time - pbar.n)

pbar.close()
process.wait()
print("Processing complete.")


Video codec: mpeg2video


Processing Video:   0%|                           | 0/342.933333 [00:25<?, ?s/s]

Processing complete.


In [4]:
#crop video 
# Input and output file paths
input_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20_rotated_30deg_padded.mp4"
output_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20_rotated_cropped_output.avi"
# Define the four vertices of the rectangle for cropping (x1, y1), (x2, y2), (x3, y3), (x4, y4)
vertices = [(51, 373), (709, 373), (709, 404), (51, 404)]


# Get the video dimensions
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])

# Calculate the bounding box for cropping
x_coordinates = [v[0] for v in vertices]
y_coordinates = [v[1] for v in vertices]
x_min, x_max = max(0, min(x_coordinates)), min(width, max(x_coordinates))
y_min, y_max = max(0, min(y_coordinates)), min(height, max(y_coordinates))

# Calculate crop dimensions
crop_x = x_min
crop_y = y_min
crop_width = x_max - x_min
crop_height = y_max - y_min

# Validate crop dimensions to ensure they are within the video frame
if crop_width <= 0 or crop_height <= 0 or crop_x + crop_width > width or crop_y + crop_height > height:
    raise ValueError(f"Invalid crop dimensions: {crop_width}x{crop_height} at position ({crop_x}, {crop_y}). "
                     f"Ensure crop area is within video bounds {width}x{height}.")

# Define the ffmpeg command to crop the video
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('crop', crop_width, crop_height, crop_x, crop_y)
    .output(output_file, vcodec='libx264', crf=23, pix_fmt='yuv420p')  # Adjust codec/compression as needed
)

# Run ffmpeg command
ffmpeg_command.run(overwrite_output=True)
print("Cropping complete.")

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.1.0.2.5)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --

Cropping complete.


[out#0/avi @ 0x6000010bc240] video:2038KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 24.427990%
frame=20576 fps=2235 q=31.0 Lsize=    2536KiB time=00:05:42.90 bitrate=  60.6kbits/s speed=37.2x    
[libx264 @ 0x134607160] frame I:83    Avg QP:26.16  size:   825
[libx264 @ 0x134607160] frame P:5185  Avg QP:28.19  size:   201
[libx264 @ 0x134607160] frame B:15308 Avg QP:31.52  size:    64
[libx264 @ 0x134607160] consecutive B-frames:  0.8%  0.1%  0.1% 99.0%
[libx264 @ 0x134607160] mb I  I16..4: 57.9%  2.9% 39.1%
[libx264 @ 0x134607160] mb P  I16..4:  7.6%  0.6%  1.5%  P16..4: 33.6%  4.4%  2.0%  0.0%  0.0%    skip:50.4%
[libx264 @ 0x134607160] mb B  I16..4:  1.3%  0.1%  0.0%  B16..8: 22.5%  1.1%  0.1%  direct: 0.9%  skip:73.9%  L0:49.8% L1:47.1% BI: 3.0%
[libx264 @ 0x134607160] 8x8 transform intra:5.9% inter:58.6%
[libx264 @ 0x134607160] coded y,uvDC,uvAC intra: 20.6% 61.1% 25.4% inter: 3.1% 10.6% 1.3%
[libx264 @ 0x134607160] i16 v,h,dc,p:  1% 98%  1%